# Request a SO-ARM101 (LeRobot)

Provision a Chameleon Edge device for SO-ARM101 robot arm teleoperation and
data collection using HuggingFace LeRobot.

## Prerequisites
- Active Chameleon Cloud allocation (project `CHI-261589`)
- A SO-ARM101 device registered on CHI@Edge
- Container image with LeRobot + teleoperation stack pushed to Docker Hub
- python-chi 1.0+ (`pip install python-chi`)

In [24]:
import chi
from datetime import timedelta

chi.use_site("CHI@Edge")
chi.set("project_name", "CHI-261589")

from chi.lease import Lease
from chi.container import Container

Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org


## Lease the device

Request (or reuse) a lease for the SO-ARM101 edge device. If an active lease
named `LEASE_NAME` already exists it will be reused — safe to re-run.

In [25]:
from chi import lease as lease_api

LEASE_NAME = "lerobot-soarm101-lease"
DEVICE_NAME = "soarm101-1"

# Check for existing active lease before creating
my_lease = None
try:
    existing = lease_api.get_lease(LEASE_NAME)
    if existing and existing.status in ("ACTIVE", "PENDING"):
        my_lease = existing
        print(f"Reusing lease '{my_lease.name}' [{my_lease.status}]")
        print(f"  ID:   {my_lease.id}")
        print(f"  Ends: {my_lease.end_date}")
except Exception:
    pass

if my_lease is None:
    print(f"No active lease named '{LEASE_NAME}'. Creating...")
    # CHI@Edge max lease duration is 7 days
    my_lease = Lease(name=LEASE_NAME, duration=timedelta(days=7))
    my_lease.add_device_reservation(device_name=DEVICE_NAME, amount=1)
    my_lease.submit(wait_for_active=True, idempotent=True)
    print(f"Lease '{my_lease.name}' is {my_lease.status} (id: {my_lease.id})")
    print(f"Ends: {my_lease.end_date}")

Reusing lease 'lerobot-soarm101-lease' [ACTIVE]
  ID:   96d5dd14-c18a-4380-b09d-b4f487bace17
  Ends: 2026-04-11 23:50:00


## Launch the container

Creates the container with camera access via the `pi_camera` device profile.
Stale/errored containers are cleaned up automatically. Safe to re-run.

### Device profile status

| Profile | Exposes | Status |
|---------|---------|--------|
| `pi_camera` | `/dev/video0`–`video23` (C920e webcam) | ✅ Used |
| `pi_serial` | `/dev/ttyACM0` only — leader arm | ⚠️ Only one port; helpdesk request pending for `ttyACM1` |

> Until the custom dual-serial profile is available, data collection requires
> running the container locally on the Pi with `--privileged -v /dev:/dev`.

In [ ]:
import time
from chi import clients
from chi.container import Container

CONTAINER_NAME = "lerobot-soarm101-container"

# ── Clean up any stale container with this name ──
zun = clients.zun()
existing = [c for c in zun.containers.list() if c.name == CONTAINER_NAME]
if existing:
    stale = existing[0]
    print(f"Found existing container [{stale.status}] — deleting...")
    Container.from_zun_container(stale).delete()
    for _ in range(12):
        time.sleep(5)
        if not any(c.name == CONTAINER_NAME for c in zun.containers.list()):
            print("  Deleted.")
            break
    else:
        raise RuntimeError("Timed out waiting for stale container to delete.")

# ── Create ──
reservation_id = my_lease.device_reservations[0]["id"]

my_container = Container(
    name=CONTAINER_NAME,
    image_ref="rianders/lerobot-soarm101:latest",
    reservation_id=reservation_id,
    environment={
        "HF_USER": "rianders",
        "LEADER_PORT": "/dev/ttyACM0",
        "FOLLOWER_PORT": "/dev/ttyACM1",
        "CAMERA_INDEX": "0",
    },
    device_profiles=["pi_camera"],  # webcam access; pi_serial pending helpdesk for dual ttyACM
)
my_container.submit(wait_for_active=True, idempotent=True)

print(f"Container '{my_container.name}' is {my_container.status}")
print(f"ID: {my_container.zun_container.uuid}")

In [ ]:
print(my_container.logs())

In [ ]:
print(f"Container '{my_container.name}' is {my_container.status}")

## Assign a floating IP

In [ ]:
# Assign floating IP if not already attached
if not getattr(my_container, 'floating_ip', None):
    my_container.associate_floating_ip()

print(f"Public IP: {my_container.floating_ip}")
print(f"\nSSH: ssh root@{my_container.floating_ip}")

## Verify the environment

Check that the camera, LeRobot, and USB devices are accessible.

In [ ]:
output, exit_code = my_container.execute("ls -l")
print(output)

In [ ]:
# Check that the SO-ARM101 serial device is visible
output, _ = my_container.execute("ls -l /dev/ttyUSB* /dev/ttyACM* 2>/dev/null || echo 'No serial devices found'")
print(output)

In [ ]:
# Check for video devices exposed by pi_camera profile
output, _ = my_container.execute("ls -l /dev/video* 2>/dev/null || echo 'No video devices found'")
print(output)

# Check v4l2 device info for the C920e
output, _ = my_container.execute("v4l2-ctl --device=/dev/video0 --info 2>/dev/null | head -10 || echo 'v4l2-ctl not available'")
print(output)

In [ ]:
# Capture a test frame from the C920e and display it here
# Saves a JPEG to /tmp inside the container, then downloads it
import io
from IPython.display import Image, display

test_cmd = """\
python3 -c "
import cv2, sys
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
ret, frame = cap.read()
cap.release()
if not ret:
    print('ERROR: could not read frame', file=sys.stderr)
    sys.exit(1)
cv2.imwrite('/tmp/test_frame.jpg', frame)
h, w = frame.shape[:2]
print(f'Captured {w}x{h} frame OK')
"
"""
output, exit_code = my_container.execute(test_cmd)
print(output)

if exit_code == 0:
    # Download the frame and display inline
    my_container.download("/tmp/test_frame.jpg", "/tmp/test_frame.jpg")
    display(Image(filename="/tmp/test_frame.jpg"))
else:
    print("Camera capture failed — check device is connected and pi_camera profile is active.")

In [ ]:
# Verify Python and LeRobot are installed
output, _ = my_container.execute("python3 --version && python3 -c 'import lerobot; print(f\"LeRobot version: {lerobot.__version__}\")'")
print(output)

## Teleoperation & Data Collection

> ⚠️ **Requires dual serial port access.** A helpdesk request for a custom
> `pi_serial` profile exposing both `ttyACM0` (leader) and `ttyACM1` (follower)
> is pending with Chameleon support. Until then, run collection locally on the
> Pi with `--privileged`.

Once the custom profile is available, add `"pi_serial"` to `device_profiles`
in the container creation cell and re-run.

### Supported training policies
- **ACT** — recommended for MI100 (no Flash Attention needed)
- **Diffusion** — heavier compute, solid results
- **SmolVLA** — lightweight VLA for constrained hardware

In [ ]:
# Record demonstration episodes — requires dual serial profile (see note above)
# Run bash scripts/collect_demos.sh inside the container instead for full control
record_cmd = """\
bash /app/scripts/collect_demos.sh soarm101_demos 20 "Pick up the object and place it in the target location"
"""
print("Starting data collection ...")
output, _ = my_container.execute(f"bash -c '{record_cmd.strip()}'")
print(output)

In [ ]:
# List collected dataset files
output, _ = my_container.execute("find /lerobot/data -type f | head -30")
print(output)

## Cleanup

Destroy the container and release the lease when finished.

In [ ]:
import time
from chi import clients

confirm = input(f"Delete container '{CONTAINER_NAME}'? [y/N]: ")
if confirm.strip().lower() in ('y', 'yes'):
    zun = clients.zun()

    # Find the container
    containers = zun.containers.list()
    target = next((c for c in containers if c.name == CONTAINER_NAME), None)

    if target is None:
        print(f"No container named '{CONTAINER_NAME}' found — already deleted?")
    else:
        print(f"Deleting {target.name} ({target.uuid}) [{target.status}]...")
        my_container.delete()

        # Poll until gone
        for i in range(12):
            time.sleep(5)
            remaining = [c for c in zun.containers.list() if c.name == CONTAINER_NAME]
            if not remaining:
                print(f"Confirmed deleted after {(i+1)*5}s.")
                break
            print(f"  {(i+1)*5}s... still {remaining[0].status}")
        else:
            print("Timed out waiting for deletion — check the portal.")
else:
    print("Cancelled.")

In [ ]:
# Remove lease when you're done with the device
# my_lease.delete()